## Module_3: *(Template)*

## Team Members:
Delaney Broderick, Priya Shan 

## Analyzing the Extent of Fibrosis at Different Lung Depths 



## Project Goal:
This project seeks to analyze a lung biopsy device developed by Intuitive, a medical device company by developing an analysis pipeline to predict the extent of fibrosis in the lung at different biopsy depths from the top of the lung. 

## Disease Background: 
*Fill in information and please note that this module is truncated and only has 5 bullets (instead of the 11 that you did for Module #1).*

* Prevalence & incidence
* Risk factors (genetic, lifestyle)
* Symptoms
* Standard of care treatment(s)
* Biological mechanisms (anatomy, organ physiology, cell & molecular physiology)

## Data-Set: 
*Describe the data set(s) you will analyze. Cite the source(s) of the data -- in this case, these data have not yet been published and they are "hot off the press", having been recently generated by the Peirce-Cottler Lab in collaboration with the Kim Lab (https://uvahealth.com/findadoctor/John-Kim-1407155682) in the Division of Pulmonary and Critical Care at UVA. So the proper way to cite these data is:

"Unpublished data was collected by the Peirce-Cottler Lab (Dept. of Biomedical Engineering) and Kim Lab (Division of Pulmonary and Critical Care) at the University of Virginia School of Medicine. 

Also, since we covered this in Lecture 1 of this Module in detail, describe how the data was collected -- What techniques were used? What units are the data measured in? Etc.*

## Data Analyis: 

General Overview: __________________________. In addition, since images of the lung are not provided at each depth, we develped an algorithm to interpolate the amount of fibrosis at a specific depth, based on the provided sample data set. 

*Describe how you analyzed the data. This is where you should intersperse your Python code so that anyone reading this Jupyter notebook can run your code to perform the analysis that you did, generate your figures, generate your .csv file, etc.). Show your graphs here, which should have proper labeles (e.g., x- and y-axes labels). Each graph you present should be thoroughly described by a caption so reader understands the data, why you are presenting it, and the main conclusion fromt the data.*

### 1. Quantifying Images of the Lung for Analysis 

This provides the foundation to analyze the extent of fibrosis at different depths in the lung. Here we follow the principle that white regions in images represent fibrotic lesions and black regions represent the healthy lung. 

In [1]:
#Importing Necessary Libraries for Analysis 

from termcolor import colored
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import pandas as pd
import time

In [2]:
# Load the images you want to analyze
#The total length of a mouse lung is around 10,000 micrometers. 
#Thus, we chose to analyze 20 images at around equal intervals apart within the lung
filenames = [
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010039.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010032.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010066.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010067.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010146.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010143.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010147.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010149.jpg", 
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010065.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010134.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010110.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010112.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010168.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010034.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010130.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010149.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010119.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010146.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Llobe ch010121.jpg",
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010135.jpg", 
    r"/Users/priya/Documents/Comp_Bio/GitHub/Module-3-Fibrosis/images/MASK_SK658 Slobe ch010098.jpg"

]

# Enter the depth of each image (in the same order that the images are listed above; you can find these in the .csv file provided to you which is tilted: "Filenames and Depths for Students")

depths = [
    15, 500, 1000, 1500, 2000, 2600, 3000, 3350, 3900, 4500,
    5300, 5500, 6000, 6600, 7000, 7500, 8000, 8500, 9000, 9500, 10000
]

In [4]:
# Results storage
results = []

# Single loop for all logic
for file, depth in zip(filenames, depths):
    #read in the image in grayscale 
    img = cv2.imread(file, 0)
    
    if img is None:
        print(colored(f"Warning: Could not load {file}", "red"))
        continue

    # Use a binary threshold
    _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    
    # Vectorized counting
    white_count = np.count_nonzero(binary)  # Faster than np.sum(binary == 255)
    total_pixels = binary.size
    black_count = total_pixels - white_count
    #determine the percentage of white pixels in the image
    white_percent = (white_count / total_pixels) * 100

    # Print results as a check point 
    #print(colored(f"File: {file}", "red"))
    #print(f"White: {white_count} | Black: {black_count}")
    #print(f"Fibrosis: {white_percent:.2f}% | Depth: {depth} microns\n")

    # Store data for CSV
    results.append({
        'Filenames': file,
        'Depths': depth,
        'White Counts': white_count,
        'White percents': white_percent
    })

### Results from the Image Analysis 

In [ ]:
# Create DataFrame and Export results to a csv 
df = pd.DataFrame(results)
df.to_csv('Percent_White_Pixels.csv', index=False)

print(colored("Success: 'Percent_White_Pixels.csv' created.", "green"))

print("The .csv file 'Percent_White_Pixels.csv' has been created.")
#print the first few rows of the dataframe to check that it looks correct
print(df.head())

Success: 'Percent_White_Pixels.csv' created.
The .csv file 'Percent_White_Pixels.csv' has been created.
                                           Filenames  Depths  White Counts  \
0  /Users/priya/Documents/Comp_Bio/GitHub/Module-...      15         21648   
1  /Users/priya/Documents/Comp_Bio/GitHub/Module-...     500         48667   
2  /Users/priya/Documents/Comp_Bio/GitHub/Module-...    1000         60715   
3  /Users/priya/Documents/Comp_Bio/GitHub/Module-...    1500         62508   
4  /Users/priya/Documents/Comp_Bio/GitHub/Module-...    2000         62913   

   White percents  
0        0.516129  
1        1.160312  
2        1.447558  
3        1.490307  
4        1.499963  


### 2. Interpolating Extent of Fibrosis at Different Depths 

In [ ]:
# Linear interpolation of data 
interpolate_depth = float(input(colored(
    "Enter the depth at which you want to interpolate a point (in microns): ", "yellow")))

x = depths
y = white_percent

# # You can also use 'quadratic', 'cubic', etc.
i = interp1d(x, y, kind='linear')
interpolate_point = i(interpolate_depth)
print(colored(
   f'The interpolated point is at the x-coordinate {interpolate_depth} and y-coordinate {interpolate_point}.', "green"))

depths_i = depths[:]
depths_i.append(interpolate_depth)
white_percents_i = white_percent[:]
white_percents_i.append(interpolate_point)

In [ ]:
#Plot the results from linear interpolation 
# make two plots: one that doesn't contain the interpolated point, just the data calculated from your images, and one that also contains the interpolated point (shown in red)
fig, axs = plt.subplots(2, 1)

axs[0].scatter(depths, white_percent, marker='o', linestyle='-', color='blue')
axs[0].set_title('Plot of depth of image vs percentage white pixels')
axs[0].set_xlabel('depth of image (in microns)')
axs[0].set_ylabel('white pixels as a percentage of total pixels')
axs[0].grid(True)


axs[1].scatter(depths_i, white_percents_i, marker='o',
               linestyle='-', color='blue')
axs[1].set_title(
   'Plot of depth of image vs percentage white pixels with interpolated point (in red)')
axs[1].set_xlabel('depth of image (in microns)')
axs[1].set_ylabel('white pixels as a percentage of total pixels')
axs[1].grid(True)
axs[1].scatter(depths_i[len(depths_i)-1], white_percents_i[len(white_percents_i)-1],
               color='red', s=100, label='Highlighted point')


# # Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

## Verify and validate your analysis: 
*Describe how you checked to see that your analysis gave you an answer that you believe (verify). Describe how your determined if your analysis gave you an answer that is supported by other evidence (e.g., by comparing your analysis to a published paper).*

## Conclusions and Ethical Implications: 
*Think about the answer your analysis generated, draw conclusions related to your overarching question, and discuss the ethical implications of your conclusions.*

## Limitations and Future Work: 
*Describe the limitations of your project. If you had more time to work on this, what would you do to explore further or refine your analysis?*

## References:
*You can use any format you like but provide the citations for facts that you referenced in this project notebook.*